# BrandMind: Intent Classification & ML Benchmarking Notebook
**DEPI Capstone Project — Team 4**

This notebook trains and benchmarks traditional ML models against the LLM intent classification baseline over the 9 specified BrandMind intents:
`shipping`, `returns`, `exchange`, `refund`, `product_info`, `availability`, `order_status`, `complaint`, `human_support`.

In [ ]:
# 1. Environment Setup
!pip install -q scikit-learn sentence-transformers pandas numpy matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sentence_transformers import SentenceTransformer

# Load Synthetic Dataset
df = pd.read_csv('../data/structured/intent_dataset.csv')
print(f"Total Samples: {len(df)}")
df['intent'].value_counts()

In [ ]:
# 2. Stratified 70/15/15 Split
X = df['text'].values
y = df['intent'].values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)
print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")

In [ ]:
# 3. Baseline 1: Majority Class
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)
print("Majority Baseline Accuracy:", accuracy_score(y_test, y_pred_dummy))
print("Majority Baseline Macro-F1:", f1_score(y_test, y_pred_dummy, average='macro', zero_division=0))

In [ ]:
# 4. Baseline 2: TF-IDF + Logistic Regression
tfidf = TfidfVectorizer(max_features=1500, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

lr_tfidf = LogisticRegression(max_iter=1000, random_state=42)
lr_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf = lr_tfidf.predict(X_test_tfidf)

print("TF-IDF + LR Accuracy:", accuracy_score(y_test, y_pred_tfidf))
print("TF-IDF + LR Macro-F1:", f1_score(y_test, y_pred_tfidf, average='macro'))

In [ ]:
# 5. Chosen Model: Sentence Embeddings (all-MiniLM-L6-v2) + Logistic Regression
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
X_train_emb = model.encode(list(X_train))
X_test_emb = model.encode(list(X_test))

lr_emb = LogisticRegression(C=1.5, max_iter=1000, random_state=42)
lr_emb.fit(X_train_emb, y_train)
y_pred_emb = lr_emb.predict(X_test_emb)

print("MiniLM + LR Accuracy:", accuracy_score(y_test, y_pred_emb))
print("MiniLM + LR Macro-F1:", f1_score(y_test, y_pred_emb, average='macro'))
print("\nDetailed Classification Report:\n", classification_report(y_test, y_pred_emb, zero_division=0))

In [ ]:
# 6. Confusion Matrix Visualization
classes = sorted(list(set(y)))
cm = confusion_matrix(y_test, y_pred_emb, labels=classes)

plt.figure(figsize=(9, 7))
plt.imshow(cm, cmap='Blues')
plt.title('BrandMind Intent Classifier Confusion Matrix')
plt.xticks(ticks=range(len(classes)), labels=classes, rotation=45, ha='right')
plt.yticks(ticks=range(len(classes)), labels=classes)
plt.xlabel('Predicted Intent')
plt.ylabel('True Intent')
plt.colorbar()
for i in range(len(classes)):
    for j in range(len(classes)):
        plt.text(j, i, cm[i, j], ha='center', va='center', color='red' if cm[i, j] > cm.max()/2 else 'black')
plt.tight_layout()
plt.show()